In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import scanpy as sc
import scvi
import pathlib
import warnings
import gc
import os
import random
warnings.filterwarnings('ignore')

sc.set_figure_params(figsize=(4, 4))

import torch
import copy as cp
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

[rank: 0] Global seed set to 0


In [2]:
print(torch.cuda.is_available)

<function is_available at 0x14e6794e5e10>


## Data loading

In [3]:
Heart = sc.read_h5ad('/rwthfs/rz/cluster/home/noco0013/projects/novo_postdoc/data/Heart/scVI_Heart_atlas.h5ad')

In [4]:
scvi.model.SCVI.setup_anndata(Heart, layer="counts", batch_key="Donor")
vaeH = scvi.model.SCVI(Heart)
vaeH = vaeH.load("/rwthfs/rz/cluster/home/noco0013/projects/novo_postdoc/data/Heart/scVI_vaeH", Heart)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


INFO     File /rwthfs/rz/cluster/home/noco0013/projects/novo_postdoc/data/Heart/scVI_vaeH/model.pt already         
         downloaded                                                                                                


In [5]:
Heart

AnnData object with n_obs × n_vars = 1822667 × 5000
    obs: 'Tissue', 'nCount_RNA', 'nFeature_RNA', 'Technology', 'Assay', 'Study', 'Sex', 'Donor', 'Age', 'Condition', 'Death', 'BMI', 'Hypertension', 'Diabetes', 'Genotype', 'Doublets', 'Cell_type', 'Subcluster', 'n_genes', 'Ischemia', 'n_umi', 'entropy', 'doublet_score', 'SNP', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', '_scvi_batch', '_scvi_labels', 'leiden02', 'leiden03', 'leiden04'
    var: 'feature_name-0', 'n_cells-0', 'name-0-1', 'n_cells-0-1', 'gene_id-0-1', 'n_cells-0-0-1-1', 'gene_symbol-0-0-1-1', 'n_cells-1-0-1-1', 'n_cells-2-0-1-1', 'gene_symbol-2-0-1-1', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'ribo', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'hvg', 'leiden', 'log1p', 'neighbor

In [6]:
cell_dict = {'Cardiomyocytes': ['0', '11', '14', '16'], 'Fibroblast': ['1'], 'Endothelial': ['2'], 'Pericytes':["3"],
             'Monocytes': ['4'], 'VSMC': ['5'], 'Lymphocytes': ['6'],'Ischemic cells (MI)':['7'], 'Endocardial':['8'], 'Neuronal':['9'], 
             'Lymphatic EC':['10'], 'Mast cells':['12'], 'Adipocytes':['13'], 'Epicardium':['15']}
Heart.obs['Annotation_1'] = np.nan
for i in cell_dict.keys():
    ind = pd.Series(Heart.obs.leiden04).isin(cell_dict[i])
    Heart.obs.loc[ind,'Annotation_1'] = i

In [7]:
cell_dict = {'Cardiomyocytes': ['0', '11', '14', '16'], 'Fibroblast': ['1'], 'Endothelial': ['2'], 'VSMC_Pericytes':["3","5"],
             'Monocytes': ['4'], 'Lymphocytes': ['6'],'Ischemic cells (MI)':['7'], 'Endocardial':['8'], 'Neuronal':['9'], 
             'Lymphatic EC':['10'], 'Mast cells':['12'], 'Adipocytes':['13'], 'Epicardium':['15']}
Heart.obs['Annotation_1'] = np.nan
for i in cell_dict.keys():
    ind = pd.Series(Heart.obs.leiden04).isin(cell_dict[i])
    Heart.obs.loc[ind,'Annotation_VP'] = i

In [8]:
np.unique(Heart.obs.Annotation_1)

array([nan])

In [9]:
Heart.X = Heart.layers["counts"]
Heart = Heart.raw.to_adata()

In [10]:
#subset = Heart[Heart.obs["Annotation_1"] == 'Fibroblast']

In [11]:
#subset.X.data

In [12]:
#gc.collect()
#subset.raw = subset
#subset.layers["counts"] = subset.X.copy()

In [13]:
#gc.collect()
#sc.pp.highly_variable_genes(
#    subset,
#    flavor="seurat_v3",
#    n_top_genes=5000,
#    layer="counts",
#    ##batch_key="Donor",
#    subset=True,
#    span=1)

In [14]:
##sc.pp.normalize_total(subset, target_sum=1e4)
##gc.collect()
##sc.pp.log1p(subset)
##sc.pp.scale(subset, max_value=10)

In [15]:
##gc.collect()
##sc.tl.pca(subset, svd_solver='arpack')

In [16]:
##scvi.model.SCVI.setup_anndata(subset, layer="counts", batch_key="Donor")
##vae_subset = scvi.model.SCVI(subset, n_layers=2, n_latent=30, gene_likelihood="nb")
##vae_subset.train()
##subset.obsm["X_scVI"] = vae_subset.get_latent_representation()
##sc.pp.neighbors(subset, use_rep="X_scVI")

In [17]:
##sc.tl.umap(subset)

In [18]:
##sc.tl.leiden(subset, resolution=0.5, key_added="leiden04")
##sc.pl.umap(
  #  subset,
  #  color=["Study", "Subcluster", "leiden04"],
  #  frameon=False,
  #  ncols=1,)

In [19]:
#subset = Heart[Heart.obs["Study"] == "Reichart 2022"]
#del subset.obs["Cell_type"]
#subset.obs.rename(columns={'cell_type': 'Cell_type'}, inplace=True)
#Heart.obs.update(subset.obs)
#del Heart.obs["cell_type"]

In [20]:
#for cluster in np.unique(Heart.obs.Annotation_1):
    #print(cluster)
    #subset = Heart[Heart.obs["Study"] == study_name]
    #subset
    #subset.write('scVI_Heart.h5ad')
    #sc.pl.umap(ref_s,
    #           color=['Cell_type'],
    #           frameon=False,
    #           wspace=0.6, s=60,
     #          title=s
     #          )

In [21]:
#Simonson = sc.read_h5ad("Simonson_2023.h5ad")
#Simonson.obs = Simonson.obs.set_index(Simonson.obs.index.astype(str) + "-1")
#Heart.obs["Subcluster"] = Simonson.obs["sub_cluster"]

In [22]:
#subset = Heart[Heart.obs["Study"] == "Chaffin 2022"]
#del subset.obs["Subcluster"]
#subset.obs.rename(columns={'SubCluster': 'Subcluster'}, inplace=True)
#Heart.obs.update(subset.obs)
#del Heart.obs["SubCluster"]
#subset = Heart[Heart.obs["Study"] == "Chaffin 2022"]
#subset.obs["SubCluster"]

In [23]:
#del Heart.obs["SubCluster"]

In [24]:
#subset = Heart[Heart.obs["Study"] == "Litvinukova 2020"]
#del subset.obs["Subcluster"]
#subset.obs.rename(columns={'cell_states': 'Subcluster'}, inplace=True)
#Heart.obs.update(subset.obs)
#subset = Heart[Heart.obs["Study"] == "Litvinukova 2020"]
#subset.obs["cell_states"]

In [25]:
#subset = Heart[Heart.obs["Study"] == "Reichart 2022"]
#del subset.obs["Subcluster"]
#subset.obs.rename(columns={'cell_states': 'Subcluster'}, inplace=True)
#Heart.obs.update(subset.obs)
#subset = Heart[Heart.obs["Study"] == "Reichart 2022"]
#subset.obs["cell_states"]

In [26]:
#del Heart.obs["cell_states"]

In [27]:
random.seed(89)
#clusters =['VSMC_Pericytes']
for cluster in np.unique(Heart.obs.Annotation_1):
for cluster in clusters:
    print(cluster)
    subset = Heart[Heart.obs["Annotation_VP"] == cluster]
    subset = subset.copy()
    gc.collect()
    subset.layers["counts"] = subset.X.copy()
    sc.pp.highly_variable_genes(
    subset,
    flavor="seurat_v3",
    n_top_genes=5000,
    layer="counts",
    ##batch_key="Donor",
    subset=False, span=1)
    scvi.model.SCVI.setup_anndata(subset, layer="counts", batch_key="Donor")
    vae_subset = scvi.model.SCVI(subset, n_layers=2, n_latent=30, gene_likelihood="nb")
    vae_subset.train()
    subset.obsm["X_scVI"] = vae_subset.get_latent_representation()
    sc.pp.neighbors(subset, use_rep="X_scVI", n_neighbors=30)
    sc.tl.leiden(subset, resolution=0.1, key_added="leiden01")
    sc.tl.leiden(subset, resolution=0.2, key_added="leiden02")
    sc.tl.leiden(subset, resolution=0.3, key_added="leiden03")
    sc.tl.leiden(subset, resolution=0.4, key_added="leiden04")
    sc.tl.leiden(subset, resolution=0.5, key_added="leiden05")
    sc.tl.leiden(subset, resolution=0.6, key_added="leiden06")
    sc.tl.leiden(subset, resolution=0.7, key_added="leiden07")
    sc.tl.umap(subset)
    file_name = cluster + '_subclustering.h5ad'
    model_name = 'vae_' + cluster
    path = '/rwthfs/rz/cluster/home/noco0013/projects/novo_postdoc/data/Heart/Subclustering_2'
    file_path = os.path.join(path, file_name)
    model_path = os.path.join(path, model_name)
    subset.write(file_path)
    vae_subset.save(model_path)

VSMC_Pericytes


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 27/27: 100%|██████████| 27/27 [13:01<00:00, 28.78s/it, loss=3.19e+03, v_num=1]

`Trainer.fit` stopped: `max_epochs=27` reached.


Epoch 27/27: 100%|██████████| 27/27 [13:01<00:00, 28.96s/it, loss=3.19e+03, v_num=1]
